# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anirban4747/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
print("""
Task type: Scoring / Ranking, built on a binary classification sub-problem.

The core predictive piece is binary classification: for each page, predict
P(page is declining) as a probability between 0 and 1.

But the deliverable isn't a single yes/no per page in isolation — it's a
RANKED QUEUE, all pages ordered by that probability (or a blended score),
so a reviewer works top-down with limited capacity. That relative ordering
is what makes this a ranking/scoring task layered on top of a classifier,
not a plain classification task judged prediction-by-prediction.
""")


Task type: Scoring / Ranking, built on a binary classification sub-problem.

The core predictive piece is binary classification: for each page, predict
P(page is declining) as a probability between 0 and 1.

But the deliverable isn't a single yes/no per page in isolation — it's a
RANKED QUEUE, all pages ordered by that probability (or a blended score),
so a reviewer works top-down with limited capacity. That relative ordering
is what makes this a ranking/scoring task layered on top of a classifier,
not a plain classification task judged prediction-by-prediction.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
print("""
Target: is_declining_label = (trend_direction == "down")

This is a PROXY label, not an observed future outcome. It comes from a
DEFINED RULE applied to the current window's trend bucket, not from
watching what actually happened after a decision point.

A stronger target for later weeks: features from a prior window -> decline
observed in a LATER window (using fact_content_daily_performance from the
warehouse), with a clean feature-window / target-window split so the label
can't leak into the features.
""")


Target: is_declining_label = (trend_direction == "down")

This is a PROXY label, not an observed future outcome. It comes from a
DEFINED RULE applied to the current window's trend bucket, not from
watching what actually happened after a decision point.

A stronger target for later weeks: features from a prior window -> decline
observed in a LATER window (using fact_content_daily_performance from the
warehouse), with a clean feature-window / target-window split so the label
can't leak into the features.



## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
print("""
Success metric: Precision@50

Reasoning: the real action is "a reviewer works through a limited list"
not "classify every page correctly" Precision@50 asks exactly the question
that matters for that action: of the top 50 pages flagged, how many are
genuinely declining? This maps directly to reviewer time wasted (false
positives) vs. reviewer time well spent, which plain accuracy or AUC don't
capture as directly.

From Week 1: hand rule Precision@50 = 0.240, random forest = 0.740.
""")


Success metric: Precision@50

Reasoning: the real action is "a reviewer works through a limited list"
not "classify every page correctly" Precision@50 asks exactly the question
that matters for that action: of the top 50 pages flagged, how many are
genuinely declining? This maps directly to reviewer time wasted (false
positives) vs. reviewer time well spent, which plain accuracy or AUC don't
capture as directly.

From Week 1: hand rule Precision@50 = 0.240, random forest = 0.740.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys
os.chdir("/content")
!rm -rf flyrank-ml-internship
!git clone https://github.com/anirban4747/flyrank-ml-internship.git
os.chdir("/content/flyrank-ml-internship")
!pip install -q -r requirements.txt

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("One row = one content page (content_id), at one snapshot in time")
print(f"Total rows: {len(df)}\n")

sample_cols = ["content_id", "client_id", "impressions_90d",
               "days_since_last_update", "avg_position", "ctr", "trend_direction"]
df[sample_cols].head(5)


df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(df["is_declining_label"].value_counts())
df[sample_cols + ["is_declining_label"]].head(5)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 132 (delta 45), reused 86 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.85 MiB | 23.33 MiB/s, done.
Resolving deltas: 100% (45/45), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.3 MB/s eta 0:00:00
One row = one content page (content_id), at one snapshot in time
Total rows: 30000

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
print("""
A fixed rule can only combine a small, hand-picked number of conditions the
way a person wrote them: fixed thresholds, no weighing of features against
each other, no interactions beyond what someone explicitly coded.

Concretely: the "stale AND visible" hand rule (days_since_last_update >= 180
and impressions_90d >= 500) flags only 17 of 30,000 pages -- an extremely
narrow slice defined by one exact threshold combination. It misses pages
that are stale but only moderately visible, or visible but not stale by
precisely 180 days -- real declining pages that just don't fit the rule's
exact shape.

The depth-2 decision tree in notebook 02 found a DIFFERENT and better split,
using impressions_90d and content_age_days together with thresholds the
model discovered from evidence -- not thresholds a person guessed. Precision
went from 0.240 (rule) to 0.740 (random forest) on the same data. That gap
is the case for ML here: the real "worth reviewing" pattern is a combination
of feature thresholds and interactions too complex for one clean
if-statement, and a model can search for that combination directly.
""")


A fixed rule can only combine a small, hand-picked number of conditions the
way a person wrote them: fixed thresholds, no weighing of features against
each other, no interactions beyond what someone explicitly coded.

Concretely: the "stale AND visible" hand rule (days_since_last_update >= 180
and impressions_90d >= 500) flags only 17 of 30,000 pages -- an extremely
narrow slice defined by one exact threshold combination. It misses pages
that are stale but only moderately visible, or visible but not stale by
precisely 180 days -- real declining pages that just don't fit the rule's
exact shape.

The depth-2 decision tree in notebook 02 found a DIFFERENT and better split,
using impressions_90d and content_age_days together with thresholds the
model discovered from evidence -- not thresholds a person guessed. Precision
went from 0.240 (rule) to 0.740 (random forest) on the same data. That gap
is the case for ML here: the real "worth reviewing" pattern is a combination
of feature threshol

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.